실습 7. 정비 판단 연결 리포트
- 복원 데이터의 추세로 설비 상태를 분류해 리포트 작성

목표
- 복원한 데이터의 추세로 설비 상태를 분류해 정비 판단 리포트를 작성

단계
- 복원한 제어출력을 이동평균으로 추세와 변화량 계산
- 변화량 기준으로 정상·주의·정비 검토 상태를 분류
- 추세·변화량·판단·추정 비율을 리포트로 출력

예상 결과
- 최종 추세 0.8·변화 0.374 → 정비 검토 필요, 추정 비율 11퍼센트

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv', encoding='utf-8')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
norm = df[['제어출력', '소입로온도']].asfreq('10s')

In [ ]:
# [정제 완료된 시계열 데이터 기반 룰 베이스 설비 보전 자동 의사결정 판정]
# 1. `clean_v.rolling(window=6)`: 보간이 잘 끝난 깨끗한 제어출력 변수를 활용해 60초(6*10s) 이동평균선(trend)을 뽑습니다.
# 2. `change = trend.iloc[-1] - trend.iloc[0]`: 전체 작동 전 구간에 걸친 추세 수치의 총 변화량을 구합니다.
# 3. 다중 if-else 조건문: 최종 요약 수치(`final > 0.55`)와 변화량 조건에 따라 '정비 검토 필요', '주의', '정상' 상태를 자동 판정합니다.
# * 누락 시점 보간 전처리부터 최종 분석과 통계치 검증, 기계적 열화 해석에 이르는 파이프라인의 완성형으로
#   스마트 팩토리에서 실시간 원격 상태 판정 스크립트를 주기적으로 백그라운드 구동하는 실무 모형을 제시합니다.
# * `min_periods=1`을 지정해 주어야 앞단의 결측(NaN) 구간에서도 정상적인 연산을 수행해 NaN 반환을 최소화합니다.

clean_v = norm['제어출력'].interpolate(method='time')
trend = clean_v.rolling(window=6, min_periods=1).mean()
final = trend.iloc[-1]; change = trend.iloc[-1] - trend.iloc[0]
status = '정비 검토 필요' if (final > 0.55 or change > 0.1) else ('주의' if change > 0.05 else '정상')

print('최종 추세:', round(final, 3), '변화량:', round(change, 3))
# 최종 추세: 0.8 변화량: 0.374
print('판단:', status)
# 판단: 정비 검토 필요
print('추정값 비율(%):', round(norm['제어출력'].isna().mean()*100, 1))
# 추정값 비율(%): 11.0

최종 추세: 0.8 변화량: 0.374
판단: 정비 검토 필요
추정값 비율(%): 11.0
